In [39]:
import math
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Dict

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv, CacheUnitMapper
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
# sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
from Common.Utils import save_training_results


In [40]:
# --- 1. CONFIGURATION & HYPERPARAMETERS (Section VII-B) ---
class Config:
    n_episodes: int = 300
    n_nodes: int = 3
    n_users: int = 1
    step_size: float = 10.00
    arrival_rate: float = 10.0  # users per second
    alpha: float = 0.5
    n_videos: int = 10
    n_gops: int = 30
    n_layers: int = 2
    n: int = 4
    m: int = 3
    n_tiles: int = n * m
    tiles_per_viewport: int = 4

    base_tile_b = 2e6 / n_tiles
    enh_tile_b = 15e6 / n_tiles

    max_capacity: float = 500e6  # 500 MB
    cache_capacity_percent: float = 0.1  # 10% of the total video size
    cache_size: int = int(n_videos * cache_capacity_percent)
    cache_capacity_b: float = (
        n_gops * n_tiles * base_tile_b +
        n_gops * tiles_per_viewport * enh_tile_b
    ) * cache_size

    # Hyperparameters for RL
    epsilon_start: float = 1.0
    epsilon_min: float = 0.005
    epsilon_decay: float = 0.987
    gamma: float = 0.99
    learning_rate: float = 1e-3
    batch_size: int = 32
    capacity: int = 10000
    window_len: int = 3  # LSTM sequence length (history window)

    h_short: int = 300   # sliding windows for popularity (Section VI-A)
    h_long: int = 1000

    r_base: float = 30.0 # PSNR reward for base layer (Section VI-C)
    r_enh: float = 10.0  # PSNR reward for enhancement layer
    penalty: float = 0.0 # fetch penalty (implicit in paper)

    # CPT parameters
    theta: float = 0.5
    lam: float = 3.7183

    @property
    def state_dim(self) -> int:
        # 10*C + 2 = (2C + 2Ck) * 2 + 2 (Section VI-A)
        return 10 * self.cache_size + 2

    @property
    def action_dim(self) -> int:
        # |A| = 5C + 1 (Section VI-B)
        return 5 * self.cache_size + 1

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DQN(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        hidden = action_dim  # = 5C + 1
        self.fc1 = nn.Linear(state_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)  # linear

class ReplayBuffer:
    def __init__(self, capacity: int = 2000):
        self.memory = deque(maxlen=capacity)
    def push(self, s, a, r, ns, d):
        self.memory.append((s, a, r, ns, d))
    def sample(self, batch_size: int):
        return random.sample(self.memory, batch_size)
    def __len__(self):
        return len(self.memory)

class DQNAgent:
    def __init__(self, cfg: Config):
        self.state_dim = cfg.state_dim
        self.action_dim = cfg.action_dim
        self.epsilon = 0.05
        self.gamma = 0.6
        self.batch_size = 32
        self.buffer = ReplayBuffer(2000)
        self.policy_net = DQN(self.state_dim, self.action_dim).to(device)
        self.target_net = DQN(self.state_dim, self.action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=1e-3)
        self.loss_fn = nn.MSELoss()
        self.nb_interval = 200  # train every 200 requests

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.policy_net(state)
        return q_values.argmax().item()
    
    def remember(self, s, a, r, ns, done):
        self.buffer.push(s, a, r, ns, done)

    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return
        batch = self.buffer.sample(self.batch_size)
        s, a, r, ns, d = zip(*batch)
        s = torch.tensor(np.stack(s), dtype=torch.float32).to(device)
        ns = torch.tensor(np.stack(ns), dtype=torch.float32).to(device)
        a = torch.tensor(a, dtype=torch.int64).to(device)
        r = torch.tensor(r, dtype=torch.float32).to(device)
        d = torch.tensor(d, dtype=torch.float32).to(device)

        q = self.policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            q_next = self.target_net(ns).max(1)[0]
            target = r + self.gamma * q_next * (1.0 - d)

        loss = self.loss_fn(q, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

In [42]:
class FeatureAdapter:
    def __init__(self, env: CacheEngineEnv, cfg: Config):
        self.env = env
        self.cfg = cfg

        self.video_hist_short = deque(maxlen=cfg.h_short)
        self.video_hist_long = deque(maxlen=cfg.h_long)
        self.tile_hist_short = deque(maxlen=cfg.h_short)
        self.tile_hist_long = deque(maxlen=cfg.h_long)

        self.video_freq_short = defaultdict(int)
        self.video_freq_long = defaultdict(int)
        self.tile_freq_short = defaultdict(int)
        self.tile_freq_long = defaultdict(int)

    def reset_history(self):
        queues = (
            self.video_hist_short,
            self.video_hist_long,
            self.tile_hist_short,
            self.tile_hist_long,
        )
        freqs = (
            self.video_freq_short,
            self.video_freq_long,
            self.tile_freq_short,
            self.tile_freq_long,
        )
        for q in queues:
            q.clear()
        for f in freqs:
            f.clear()

    def update_history(self, request: Dict):
        vid = request["video"]
        
        self._update_window(self.video_hist_short, self.video_freq_short, vid)
        self._update_window(self.video_hist_long, self.video_freq_long, vid)

        tiles = request["viewport"]
        tiles = tuple(tiles.tolist()) if tiles is not None else None
        
        self._update_window(self.tile_hist_short, self.tile_freq_short, tiles)
        self._update_window(self.tile_hist_long, self.tile_freq_long, tiles)
        
    def _update_window(self, hist_queue: deque, freq_dict: Dict, item):
        if item is None:
            return
        
        if len(hist_queue) == hist_queue.maxlen:
            old_item = hist_queue.popleft()
            freq_dict[old_item] -= 1
            if freq_dict[old_item] == 0:
                del freq_dict[old_item]
        hist_queue.append(item)
        freq_dict[item] += 1

In [ ]:
class NetworkAdapter:
    def __init__(self, env: EnvWrapper, feature_adapter: FeatureAdapter, cfg: Config):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter

        self.capacity = int(
            self.cfg.n_videos * self.cfg.cache_capacity_percent * self.cfg.n_gops * self.cfg.n_tiles +
            self.cfg.n_videos * self.cfg.cache_capacity_percent * self.cfg.n_gops * self.cfg.tiles_per_viewport            
        )
        self.n_features = self.capacity + 1

        self.C = self.cfg.cache_size               # paper’s cache capacity (videos)
        self.k = self.env.mec_cache.get_viewport_tile_budget()

        print(f"NetworkAdapter initialized with capacity: {self.capacity}")

    def _rank_cached_videos(self, bitmap: np.ndarray) -> list[int]:
        cached_mask = np.any(bitmap[:, 0, :, :], axis=(1, 2))
        cached_vids = np.where(cached_mask)[0].tolist()
        ranked = sorted(
            cached_vids,
            key=lambda vid: self.features.video_freq_long.get(vid, 0),
            reverse=True
        )

        pad = [-1] * (self.capacity - len(ranked))
        return ranked + pad

    def _cached_viewport_tiles(self, bitmap: np.ndarray, vid: int, k: int) -> list[int]:
        if vid == -1:
            return [-1] * k
        tile_mask = np.any(bitmap[vid, 1, :, :], axis=1)
        tiles = np.flatnonzero(tile_mask).tolist()
        tiles.extend([-1] * max(0, k - len(tiles)))
        return tiles[:k]

    def build_observation(self, request: Dict) -> np.ndarray:
        """
        Builds the state vector as in the paper:
          [ x_s (C), y_s (C*k), z_s (1), x_l (C), y_l (C*k), z_l (1) ]
        where:
          - x_s/x_l: counts of requests for cached base videos (short/long windows)
          - y_s/y_l: counts of requests for cached enh tiles per video (short/long)
          - z_s/z_l: counts for the currently examined item (video or tile)
        Total dim = 10*C + 2 when k=4.
        """        
        cache_bitmap = self.env.mec_cache.get_cache_bitmap()
        C, k = self.C, self.k

        ranked_cached = self._rank_cached_videos(cache_bitmap)[:C]

        # Initialize feature blocks
        x_s = np.zeros(C, dtype=np.float32)
        x_l = np.zeros(C, dtype=np.float32)
        y_s = np.zeros(C * k, dtype=np.float32)
        y_l = np.zeros(C * k, dtype=np.float32)

        # Fill x and y
        for i, vid in enumerate(ranked_cached):
            if vid == -1:
                continue

            x_s[i] = self.features.video_freq_short.get(vid, 0)
            x_l[i] = self.features.video_freq_long.get(vid, 0)

            tile_mask = np.any(cache_bitmap[vid, 1, :, :], axis=1)
            tiles = np.flatnonzero(tile_mask).tolist()
            tiles = tiles[:k] + [-1] * max(0, k - len(tiles))
            base_off = i * k

            for j, t in enumerate(tiles):
                if t == -1:
                    continue
                y_s[base_off + j] = self.features.tile_freq_short.get(t, 0)
                y_l[base_off + j] = self.features.tile_freq_long.get(t, 0)

        self.n_features = self.capacity + 1
        print(f"Building observation with capacity: {self.capacity}, k: {k}, n_features: {self.n_features}")
        print(len(ranked_cached), "-", ranked_cached)
        
        vid_req = request["video"]
        viewport = request["viewport"] if request["viewport"] is not None else []
        is_base = request.get("layer", 0) == 0

        z_s = np.array([
            self.features.video_freq_short.get(vid_req, 0) if is_base
            else sum(self.features.tile_freq_short.get(t, 0) for t in viewport)
        ], dtype=np.float32)
        z_l = np.array([
            self.features.video_freq_long.get(vid_req, 0) if is_base
            else sum(self.features.tile_freq_long.get(t, 0) for t in viewport)
        ], dtype=np.float32)

        return np.concatenate([x_s, y_s, z_s, x_l, y_l, z_l], axis=0)
    
    def evict_video(self, bitmap: np.ndarray, v: int):
        for layer, tile_id, gop_id in np.argwhere(bitmap[v] == 1):
            key = (v, int(layer), int(tile_id), int(gop_id))
            self.env.mec_cache.policy.remove(key)
            bitmap[v, layer, tile_id, gop_id] = 0

    def apply_action(self, action_idx: int, request: Dict) -> Dict:
        """
        Implements the paper's action space:
          - A1 (size C+1): when base is not cached. a0 = no-op; a_i evicts the i-th cached video and caches the requested one.
          - A2 (size k+1): when base is cached but viewport differs. a0 = no-op; a_j replaces the j-th cached enh tile with the j-th requested tile.
        """
        vid = request["video"]
        gop = request["gop"]
        # viewport = request["viewport"] or []

        bitmap = self.env.mec_cache.get_cache_bitmap()
        ranked_videos = self._rank_cached_videos(bitmap)
        
        C, k = self.C, self.k


        if gop == 0:
            base_cached = np.any(bitmap[vid, 0, :, :])
            
            if base_cached or action_idx == 0:
                return  # no-op
            elif 1 <= action_idx <= self.C:
                rank_idx = action_idx - 1 # 0-based index
                victim_vid = ranked_videos[rank_idx]

                if victim_vid != -1:
                    self.evict_video(bitmap, victim_vid)
                
                # Cache the new video (Base Layer)
                self.env.mec_cache.cache_new_video(vid, gop, layer=0)
            else:
                # Action index > C (Tile actions) are invalid for Base Layer decisions.
                # Treat as No-Op or Penalize in Reward function.
                return
        elif gop > 0:
            current_tile = request["viewport"]
            
            base_cached = np.any(bitmap[vid, 0, :, :])
            if not base_cached:
                return # Cannot cache tiles if base video is not present
            
            # Identify which Rank the requested video occupies
            vid_rank = self._get_video_rank(ranked_videos, vid)
            
            if vid_rank == -1:
                # Should not happen if base_cached is True, unless ranking logic desyncs
                return
            
            # Valid Action Space for Tile Layer:
            # 0: No-Op (Don't cache this tile)
            # Specific range for this video's rank: [Start, End]
            
            # Calculate the action indices belonging to this specific video rank
            # Offset = 1 (No-Op) + C (Video Actions)
            start_idx = 1 + self.C + (vid_rank * self.k)
            end_idx = start_idx + self.k
            
            if action_idx == 0:
                return # Decision: Do not cache this tile
            elif start_idx <= action_idx < end_idx:
                # The agent decided to replace a specific tile slot IN THIS VIDEO
                # slot_idx is 0..k-1 (e.g., 0..3)
                slot_idx = action_idx - start_idx
                
                # Find the actual tile ID currently occupying this slot
                # We need the list of tiles currently in the virtual viewport for this video
                current_viewport_tiles = self._cached_viewport_tiles(bitmap, vid, self.k)
                victim_tile_id = current_viewport_tiles[slot_idx]
                
                # Evict the victim tile if it exists
                if victim_tile_id != -1:
                    key = (vid, 1, victim_tile_id, gop)
                    # Assuming env policy removal works like this:
                    if key in self.env.mec_cache.policy:
                         self.env.mec_cache.policy.remove(key)
                    bitmap[vid, 1, victim_tile_id, gop] = 0
                
                # Cache the NEW candidate tile in high quality
                if current_tile is not None:
                    self.env.mec_cache.cache_new_video(vid, gop, layer=1, tile_idx=current_tile)
            
            else:
                # Agent selected an action belonging to a DIFFERENT video rank.
                # This is an invalid move for the current context. Treat as No-Op.
                return
                
        # base_cached = np.any(bitmap[vid, 0, :, :])
        # enh_cached = all(bitmap[vid, 1, t, gop] == 1 for t in viewport)

        # # Block A1: video not cached in base
        # if not base_cached:
        #     if action_idx == 0:
        #         return  # no-op
        #     else:
        #         for i in range(1, C + 1):
        #             if action_idx == i:
        #                 continue
        #             ranked = self._rank_cached_videos(bitmap)
        #             victim = ranked[i - 1]
        #             if victim != -1:
        #                 self.evict_video(bitmap, victim)
        #             self.env.mec_cache.cache_new_video(vid, gop, layer=0)

        # # Block A2: base cached; check viewport
        # if enh_cached:
        #     return  # already fully cached, no action
        # else:
        #     offset = C + 1
        #     if action_idx == offset:
        #         return  # no-op
        #     else:
        #         for j in range(1, k + 1):
        #             if action[offset + j] == 0:
        #                 continue
        #             ranked_tiles = self._cached_viewport_tiles(bitmap, vid, k)
        #             victim_tile = ranked_tiles[j - 1]
        #             if victim_tile != -1:
        #                 key = (vid, 1, victim_tile, gop)
        #                 bitmap[vid, 1, victim_tile, gop] = 0
        #             requested_tile = viewport[j - 1] if j - 1 < len(viewport) else None
        #             if requested_tile is not None:
        #                 self.env.mec_cache.cache_new_video(vid, gop, layer=1, tile_idx=requested_tile)

In [44]:
if __name__ == "__main__":
    print("--- Starting DRL Caching System ---")

    # 1. Load Configuration
    cfg = Config()
    
    # 2. Initialize Environment
    du_caches = []

    unit_mapper = CacheUnitMapper(
        cache_capacity_mb=cfg.cache_capacity_b / 1e6,
        num_gops=cfg.n_gops,
        num_tiles=cfg.n_tiles,
        viewport_tiles=4,  # assuming viewport with 4 tiles
        base_tile_mb=2e6 / 1e6 / cfg.n_tiles,
        enh_tile_mb=15e6 / 1e6 / cfg.n_tiles
    )
    
    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity_b,
        # policy=SvcLruPolicy(max_size=max_capacity)
        unit_mapper=unit_mapper
    )

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=cfg.arrival_rate,
        alpha=cfg.alpha
    )

    P = cfg.n_nodes; max_U = cfg.n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,  # 640 Mbps -> 80e6 B/s
        R_C_M=125e6, # 1 Gbps -> 125e6 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),    # 160 Mbps -> 20e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,  # 1 ms
        mec_fixed_delay=0.005, # 5 ms
        cloud_fixed_delay=0.1  # 100 ms
    )

    env = EnvWrapper(
        n=cfg.n,
        n_layers=cfg.n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        theta=cfg.theta,
        lam=cfg.lam
    )

    # 3. Initialize Agent
    agent = DQNAgent(cfg)

    obs, info = env.reset()
    feature_adapter = FeatureAdapter(env, cfg)
    net_adapter = NetworkAdapter(env, feature_adapter, cfg)

    print(
        f"->> "
        f"{mec_cache.unit_mapper.max_units} "
        f"{mec_cache.max_capacity}"
    )

    for step in count():
        reqs_state = info['users_requests']
        active_users = [
            req  for req in reqs_state if req['gop'] < cfg.n_gops
        ]
    
        for req in active_users:
            feature_adapter.update_history(req)

        actions = []
        for user_req in active_users:
            video = user_req['video']
            tiles = user_req['tiles']
            gop = user_req['gop']

            state_vec = net_adapter.build_observation(user_req)
            np.set_printoptions(threshold=np.inf, linewidth=200)
            print(state_vec.shape)   # (480004,)
            print(len(state_vec), "-", state_vec)
            
            # action_idx = np.random.randint(0, net_adapter.capacity * 5 + 1)
            action_idx = agent.select_action(state_vec)
            net_adapter.apply_action(action_idx, user_req)

        obs, rewards, done, info = env.step([])

        step_reward = float(rewards) / max(1, len(active_users))

        # --- TRAINING / HISTORY UPDATE ---
        reqs_next_state = info['users_requests']

        for user_req in active_users:

            video = user_req['video']
            tiles = user_req['tiles']
            gop = user_req['gop']
            next_state_vec = net_adapter.build_observation(user_req)

            agent.remember(state_vec, action_idx, step_reward, next_state_vec, done)
            if step % agent.nb_interval == 0:
                agent.train_step()
                agent.update_target()

        print(f"Step {step}, Active Users: {len(active_users)}")
        print(f"Request State: {reqs_state}")
        print(f"Action: {actions}")
        print(f"Next Request State: {reqs_next_state}")
        print("-----")

        if done:
            break

--- Starting DRL Caching System ---
NetworkAdapter initialized with capacity: 480
->> 1 210000000.0
Step 0, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 1, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 2, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 3, Active Users: 0
Request State: []
Action: []
Next Request State: [{'gop': 0, 'u': 0, 'p': 1, 'layer': 0, 'video': 1, 'viewport': None, 'tiles': []}]
-----
Building observation with capacity: 480, k: 4, n_features: 481
1 - [-1]
(12,)
12 - [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1.]
Building observation with capacity: 480, k: 4, n_features: 481
1 - [-1]
Step 4, Active Users: 1
Request State: [{'gop': 0, 'u': 0, 'p': 1, 'layer': 0, 'video': 1, 'viewport': None, 'tiles': []}]
Action: []
Next Request State: [{'gop': 1, 'u': 0, 'p': 1, 'video': 1, 'viewport': array([ 5,  6,  9, 10]), 'tiles': [{'tile': 0, 'layer': 0, 'size': 166666.66666666666, 'even

In [45]:
import itertools
import numpy as np

def build_action_space(C: int, k: int):
    # A1: base-not-cached actions (0 = no-op, 1..C evict ranked i-th video)
    A1 = list(range(C + 1))
    
    # A2: enhance-tile actions (C*k entries, each replaces one tile position)
    A2 = list(range(C * k))
    
    # Cartesian product (indices)
    A = list(itertools.product(A1, A2))
    
    return A1, A2, A  # A has size (C+1) * (C*k) = 5C+1 when k=4

# Example usage
C, k = 2, 4  # cache can hold 2 videos; viewport budget k=4
A1, A2, A = build_action_space(C, k)
print("A1 (C+1):", A1)
print("A2 (C*k):", A2)
print("Total |A|:", len(A), "== (C+1)*C*k =", (C+1)*C*k)

# One-hot encoding helper for the flat index in [0, 5C]
def one_hot_action(idx: int, C: int, k: int):
    size = (C + 1) + C * k  # 5C+1 when k=4
    vec = np.zeros(size, dtype=np.float32)
    vec[idx] = 1.0
    
    return vec

# Example: pick A1 action i=1 (evict first video) and A2 action m=3 (replace tile slot 3)
a1_choice, a2_choice = 1, 3
flat_idx = a1_choice if a1_choice <= C else (C + 1 + a2_choice)
action_vec = one_hot_action(flat_idx, C, k)

print("Flat index:", flat_idx)
print("One-hot vector:", action_vec)

A1 (C+1): [0, 1, 2]
A2 (C*k): [0, 1, 2, 3, 4, 5, 6, 7]
Total |A|: 24 == (C+1)*C*k = 24
Flat index: 1
One-hot vector: [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
